In [208]:
import pandas as pd

all_2247 = pd.read_csv("../Data/ST-2247.csv")
all_2252 = pd.read_csv("../Data/ST-2252.csv")
all_2425 = pd.read_csv("../Data/ST-2425.csv")

In [209]:
# 필요없는 컬럼 제거

def drop_unnamed_columns(df):
    cols_to_drop = [c for c in df.columns if "Unnamed" in str(c)]
    return df.drop(columns=cols_to_drop, inplace=True)


for df in [all_2247, all_2252, all_2425]:
    drop_unnamed_columns(df)

In [210]:
all_2247

,기준_날짜,시작_대여소_ID,시작_대여소명,종료_대여소_ID,종료_대여소명,전체_건수,전체_이용_분,전체_이용_거리,대여소_ID,주소1,...,경도,이용기준,이용시각,요일,주말,날씨시각,기온,강수량,적설량,풍속
0,20240101,ST-1036,역촌동_001_1,ST-2247,신사2동_013_1,1.0,74.0,740.0,ST-1036,서울특별시 은평구 연서로 124,...,126.916794,도착시간,2024-01-01 16:40:00,월,주말,2024-01-01 16:00:00,5.9,0.0,0.0,10.8
1,20240101,ST-463,증산동_004_1,ST-2247,신사2동_013_1,1.0,8.0,1268.0,ST-463,서울특별시 은평구 증산동 199-8,...,126.909897,도착시간,2024-01-01 19:13:00,월,주말,2024-01-01 19:00:00,0.2,0.0,0.0,8.0
2,20240101,ST-2247,신사2동_013_1,ST-2247,신사2동_013_1,1.0,11.0,1196.0,ST-2247,서울특별시 은평구 신사동 178-9,...,126.906082,도착시간,2024-01-01 18:16:00,월,주말,2024-01-01 18:00:00,2.0,0.0,0.0,7.6
3,20240101,ST-455,응암1동_032_1,ST-2247,신사2동_013_1,2.0,44.0,4070.0,ST-455,서울특별시 은평구 은평로 85 CJ드림시티,...,126.916985,도착시간,2024-01-01 22:30:00,월,주말,2024-01-01 22:00:00,-0.1,0.0,0.0,6.2
4,20240101,ST-455,응암1동_032_1,ST-2247,신사2동_013_1,2.0,44.0,4070.0,ST-455,서울특별시 은평구 은평로 85 CJ드림시티,...,126.916985,도착시간,2024-01-01 22:49:00,월,주말,2024-01-01 22:00:00,-0.1,0.0,0.0,6.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25708,20241231,ST-463,증산동_011_1,ST-2247,신사2동_013_1,1.0,8.0,1144.0,ST-463,서울특별시 은평구 증산동 199-8,...,126.909897,도착시간,2024-12-31 14:33:00,화,주중,2024-12-31 14:00:00,1.8,0.0,0.0,12.5
25709,20241231,ST-2247,신사2동_013_1,ST-2537,상암동_029_1,1.0,15.0,2141.0,ST-2247,서울특별시 은평구 신사동 178-9,...,126.906082,출발시간,2024-12-31 15:30:00,화,주중,2024-12-31 15:00:00,1.8,0.0,0.0,12.4
25710,20241231,ST-2247,신사2동_013_1,ST-2240,이촌2동_012_1,1.0,65.0,13623.0,ST-2247,서울특별시 은평구 신사동 178-9,...,126.906082,출발시간,2024-12-31 10:00:00,화,주중,2024-12-31 10:00:00,-0.7,0.0,0.0,12.2
25711,20241231,ST-2825,응암2동_050_1,ST-2247,신사2동_013_1,1.0,12.0,1545.0,ST-2825,서울특별시 은평구 은평구 응암로 278 맞은편(응암동 767-1 공원 앞),...,126.919624,도착시간,2024-12-31 07:52:00,화,주중,2024-12-31 07:00:00,-3.5,0.0,0.0,13.0


In [211]:
# 상대 대수 구하기
import pandas as pd

def build_relative_count_df(
    df,
    time_col="날씨시각",
    usage_col="이용기준",
    count_col="전체_건수",
    arrival_value="도착시간",
    depart_value="출발시간",
    result_col="상대대수",
):
    # 상대대수 계산
    signed = df[count_col].where(df[usage_col] == arrival_value,
                                 -df[count_col])
    df[result_col] = signed

    # 유지할 컬럼
    meta_cols = ["주소1", "요일", "주말", "기온", "강수량", "적설량", "풍속"]

    # 그룹바이: 상대대수는 합, 나머지는 대표값
    agg_map = {result_col: "sum"}
    for c in meta_cols:
        agg_map[c] = "first"

    out = (
        df.groupby(time_col, as_index=False)
          .agg(agg_map)
          .sort_values(time_col)
          .reset_index(drop=True)
    )

    return out


all_2247 = build_relative_count_df(all_2247)
all_2252 = build_relative_count_df(all_2252)
all_2425 = build_relative_count_df(all_2425)


In [212]:
all_2247

,날씨시각,상대대수,주소1,요일,주말,기온,강수량,적설량,풍속
0,2024-01-01 00:00:00,-2.0,서울특별시 은평구 신사동 178-9,월,주말,-1.9,0.0,0.0,6.3
1,2024-01-01 04:00:00,-2.0,서울특별시 은평구 신사동 178-9,월,주말,-2.8,0.0,0.0,7.6
2,2024-01-01 14:00:00,2.0,서울특별시 은평구 응암동 604-5,월,주말,6.4,0.0,0.0,9.4
3,2024-01-01 16:00:00,4.0,서울특별시 은평구 연서로 124,월,주말,5.9,0.0,0.0,10.8
4,2024-01-01 18:00:00,2.0,서울특별시 은평구 신사동 178-9,월,주말,2.0,0.0,0.0,7.6
...,...,...,...,...,...,...,...,...,...
5399,2024-12-31 18:00:00,-2.0,서울특별시 은평구 신사동 178-9,화,주중,-1.2,0.0,0.0,6.5
5400,2024-12-31 19:00:00,2.0,서울특별시 은평구 신사동 178-9,화,주중,-1.9,0.0,0.0,6.3
5401,2024-12-31 20:00:00,2.0,서울특별시 은평구 신사동 352-2,화,주중,-2.5,0.0,0.0,4.7
5402,2024-12-31 21:00:00,-2.0,서울특별시 은평구 신사동 178-9,화,주중,-2.9,0.0,0.0,4.1


In [213]:
# 현재 대수 구하기
import pandas as pd

def add_current_count_with_reset(
    df,
    time_col="날씨시각",
    relative_col="상대대수",
    current_col="현재대수",
    reset_hours=(4, 16),
):
    out = df.sort_values(time_col).reset_index(drop=True).copy()

    dt = pd.to_datetime(out[time_col])
    hour = dt.dt.hour

    # 04:00~15:59는 shift=0, 16:00~03:59는 shift=1로 묶기
    shift = (hour >= reset_hours[1]).astype(int)

    # 04:00 기준으로 날짜를 맞추기 위해 4시간 빼고 날짜 그룹 생성
    day_bucket = (dt - pd.Timedelta(hours=reset_hours[0])).dt.date

    group_key = day_bucket.astype(str) + "_" + shift.astype(str)

    out[current_col] = out.groupby(group_key)[relative_col].cumsum()
    return out

all_2247 = add_current_count_with_reset(all_2247)
all_2252 = add_current_count_with_reset(all_2252)
all_2425 = add_current_count_with_reset(all_2425)

----
#### 출퇴근시간, 계절 컬럼 추가

In [214]:
# 출퇴근시간, 계절 컬럼 추가
import pandas as pd

def add_commute_and_season(
    df,
    time_col="날씨시각",
    commute_col="출퇴근시간",
    season_col="계절",
    commute_hours=((7, 9), (17, 19)),
):
    out = df.copy()
    dt = pd.to_datetime(out[time_col])
    hour = dt.dt.hour

    is_commute = False
    for start, end in commute_hours:
        is_commute |= (hour >= start) & (hour <= end)
    out[commute_col] = is_commute.astype(int)

    month = dt.dt.month
    season = pd.Series(index=out.index, dtype="Int64")
    season.loc[month.isin([3, 4, 5])] = 1  # 봄
    season.loc[month.isin([6, 7, 8])] = 2  # 여름
    season.loc[month.isin([9, 10, 11])] = 3  # 가을
    season.loc[month.isin([12, 1, 2])] = 4  # 겨울
    out[season_col] = season

    return out

all_2247 = add_commute_and_season(all_2247)
all_2252 = add_commute_and_season(all_2252)
all_2425 = add_commute_and_season(all_2425)


In [215]:
# 날씨시각에서 시각만 추출
import pandas as pd

def add_time_only(df, time_col="날씨시각", time_only_col="시각"):
    out = df.copy()
    dt = pd.to_datetime(out[time_col])
    out[time_only_col] = dt.dt.hour
    return out

all_2247 = add_time_only(all_2247)
all_2252 = add_time_only(all_2252)
all_2425 = add_time_only(all_2425)


In [216]:
# 요일 원-핫 인코딩
import pandas as pd

def add_weekday_onehot(df, col="요일"):
    out = df.copy()
    dummies = pd.get_dummies(out[col], prefix=col, dtype=int)
    out = pd.concat([out, dummies], axis=1)
    return out

all_2247 = add_weekday_onehot(all_2247)
all_2252 = add_weekday_onehot(all_2252)
all_2425 = add_weekday_onehot(all_2425)


In [ ]:
# bus_stop(X,Y)에서 따릉이 정류소 반경 200m 필터링
import numpy as np
import pandas as pd

def haversine_m(lat1, lon1, lat2, lon2):
    # lat/lon in degrees
    r = 6371000  # meters
    lat1 = np.deg2rad(lat1)
    lon1 = np.deg2rad(lon1)
    lat2 = np.deg2rad(lat2)
    lon2 = np.deg2rad(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return r * c

# 따릉이 정류소 좌표 추출
bike_df = pd.read_csv("Data/찬솔님데이터.csv")
target_ids = ["ST-2247", "ST-2252", "ST-2425"]
station_coords = (
    bike_df[bike_df["대여소_ID"].isin(target_ids)]
    .groupby("대여소_ID", as_index=False)
    .agg({"대여소_위도": "median", "대여소_경도": "median"})
)

# bus_stop: X=경도, Y=위도 가정
bus_stop = bus_stop.copy()
bus_stop["_bus_lat"] = bus_stop["Y"]
bus_stop["_bus_lon"] = bus_stop["X"]

bus_stops_within_200m = {}
for _, row in station_coords.iterrows():
    sid = row["대여소_ID"]
    lat = row["대여소_위도"]
    lon = row["대여소_경도"]
    dist = haversine_m(bus_stop["_bus_lat"].values, bus_stop["_bus_lon"].values, lat, lon)
    mask = dist <= 200
    bus_stops_within_200m[sid] = bus_stop.loc[mask].copy()

bus_stops_within_200m


In [217]:
all_2247

,날씨시각,상대대수,주소1,요일,주말,기온,강수량,적설량,풍속,현재대수,출퇴근시간,계절,시각,요일_금,요일_목,요일_수,요일_월,요일_일,요일_토,요일_화
0,2024-01-01 00:00:00,-2.0,서울특별시 은평구 신사동 178-9,월,주말,-1.9,0.0,0.0,6.3,-2.0,0,4,0,0,0,0,1,0,0,0
1,2024-01-01 04:00:00,-2.0,서울특별시 은평구 신사동 178-9,월,주말,-2.8,0.0,0.0,7.6,-2.0,0,4,4,0,0,0,1,0,0,0
2,2024-01-01 14:00:00,2.0,서울특별시 은평구 응암동 604-5,월,주말,6.4,0.0,0.0,9.4,0.0,0,4,14,0,0,0,1,0,0,0
3,2024-01-01 16:00:00,4.0,서울특별시 은평구 연서로 124,월,주말,5.9,0.0,0.0,10.8,4.0,0,4,16,0,0,0,1,0,0,0
4,2024-01-01 18:00:00,2.0,서울특별시 은평구 신사동 178-9,월,주말,2.0,0.0,0.0,7.6,6.0,1,4,18,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5399,2024-12-31 18:00:00,-2.0,서울특별시 은평구 신사동 178-9,화,주중,-1.2,0.0,0.0,6.5,-4.0,1,4,18,0,0,0,0,0,0,1
5400,2024-12-31 19:00:00,2.0,서울특별시 은평구 신사동 178-9,화,주중,-1.9,0.0,0.0,6.3,-2.0,1,4,19,0,0,0,0,0,0,1
5401,2024-12-31 20:00:00,2.0,서울특별시 은평구 신사동 352-2,화,주중,-2.5,0.0,0.0,4.7,0.0,0,4,20,0,0,0,0,0,0,1
5402,2024-12-31 21:00:00,-2.0,서울특별시 은평구 신사동 178-9,화,주중,-2.9,0.0,0.0,4.1,-2.0,0,4,21,0,0,0,0,0,0,1


In [218]:
all_2247.to_csv("../Data/ST-2247_current.csv", index=False)
all_2252.to_csv("../Data/ST-2252_current.csv", index=False)
all_2425.to_csv("../Data/ST-2425_current.csv", index=False)

--------
#### 전처리

In [219]:
# 주말 인덱싱
def holiday_indexing(df):
    df["주말"] = df["주말"].map({"주말": 1, "주중": 0}).fillna(df["주말"]).astype(int)
    return df

all_2247 = holiday_indexing(all_2247)
all_2252 = holiday_indexing(all_2252)
all_2425 = holiday_indexing(all_2425)

In [220]:
all_2247

,날씨시각,상대대수,주소1,요일,주말,기온,강수량,적설량,풍속,현재대수,출퇴근시간,계절,시각,요일_금,요일_목,요일_수,요일_월,요일_일,요일_토,요일_화
0,2024-01-01 00:00:00,-2.0,서울특별시 은평구 신사동 178-9,월,1,-1.9,0.0,0.0,6.3,-2.0,0,4,0,0,0,0,1,0,0,0
1,2024-01-01 04:00:00,-2.0,서울특별시 은평구 신사동 178-9,월,1,-2.8,0.0,0.0,7.6,-2.0,0,4,4,0,0,0,1,0,0,0
2,2024-01-01 14:00:00,2.0,서울특별시 은평구 응암동 604-5,월,1,6.4,0.0,0.0,9.4,0.0,0,4,14,0,0,0,1,0,0,0
3,2024-01-01 16:00:00,4.0,서울특별시 은평구 연서로 124,월,1,5.9,0.0,0.0,10.8,4.0,0,4,16,0,0,0,1,0,0,0
4,2024-01-01 18:00:00,2.0,서울특별시 은평구 신사동 178-9,월,1,2.0,0.0,0.0,7.6,6.0,1,4,18,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5399,2024-12-31 18:00:00,-2.0,서울특별시 은평구 신사동 178-9,화,0,-1.2,0.0,0.0,6.5,-4.0,1,4,18,0,0,0,0,0,0,1
5400,2024-12-31 19:00:00,2.0,서울특별시 은평구 신사동 178-9,화,0,-1.9,0.0,0.0,6.3,-2.0,1,4,19,0,0,0,0,0,0,1
5401,2024-12-31 20:00:00,2.0,서울특별시 은평구 신사동 352-2,화,0,-2.5,0.0,0.0,4.7,0.0,0,4,20,0,0,0,0,0,0,1
5402,2024-12-31 21:00:00,-2.0,서울특별시 은평구 신사동 178-9,화,0,-2.9,0.0,0.0,4.1,-2.0,0,4,21,0,0,0,0,0,0,1


------
#### 머신러닝

In [221]:
# !pip install scikit-learn

In [ ]:
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def train_rf_with_search_time_split(
    df,
    time_col="날씨시각",
    feature_cols=("시각", "요일_월", "요일_화", "요일_수", "요일_목", "요일_금", "요일_토", "요일_일", "주말", "출퇴근시간", "계절", "기온", "강수량", "적설량", "풍속"),
    target_col="현재대수",
    test_size=0.2,
    val_size=0.2,
    random_state=42,
):
    # 1) 시간순 정렬
    df_sorted = df.sort_values(time_col).reset_index(drop=True)

    X = df_sorted[list(feature_cols)]
    y = df_sorted[target_col]

    n = len(df_sorted)
    n_test = int(n * test_size)
    n_val = int(n * val_size)

    n_train = n - n_val - n_test
    if n_train <= 0:
        raise ValueError("train/val/test 비율이 너무 큽니다.")

    # 2) 시간순 분할
    X_train, y_train = X.iloc[:n_train], y.iloc[:n_train]
    X_val, y_val = X.iloc[n_train:n_train+n_val], y.iloc[n_train:n_train+n_val]
    X_test, y_test = X.iloc[n_train+n_val:], y.iloc[n_train+n_val:]

    # 3) 파이프라인 (정규화 + 랜덤포레스트)
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", RandomForestRegressor(random_state=random_state, n_jobs=-1))
    ])

    # 4) 하이퍼파라미터 탐색
    param_grid = {
        "model__n_estimators": [200, 400],
        "model__max_depth": [None, 8, 12],
        "model__min_samples_split": [2, 5],
        "model__min_samples_leaf": [1, 2],
        "model__max_features": ["sqrt", "log2"]
    }

    search = GridSearchCV(
        pipe,
        param_grid=param_grid,
        cv=3,
        scoring="r2",
        n_jobs=-1
    )
    search.fit(X_train, y_train)

    best_model = search.best_estimator_

    # 5) 점수 출력
    def _eval(name, X_split, y_split):
        pred = best_model.predict(X_split)
        return {
            "split": name,
            "r2": r2_score(y_split, pred),
            "mae": mean_absolute_error(y_split, pred),
            "rmse": np.sqrt(mean_squared_error(y_split, pred))
        }

    scores = {
        "train": _eval("train", X_train, y_train),
        "val": _eval("val", X_val, y_val),
        "test": _eval("test", X_test, y_test),
        "best_params": search.best_params_
    }

    return best_model, scores


In [223]:
print(train_rf_with_search_time_split(all_2247))

(Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 RandomForestRegressor(max_depth=8, max_features='sqrt',
                                       min_samples_leaf=2, n_estimators=200,
                                       n_jobs=-1, random_state=42))]), {'train': {'split': 'train', 'r2': 0.40835406800050145, 'mae': 4.325288024049654, 'rmse': np.float64(5.903353979922953)}, 'val': {'split': 'val', 'r2': 0.01584746965279682, 'mae': 6.46069825940774, 'rmse': np.float64(8.909493256685975)}, 'test': {'split': 'test', 'r2': 0.15710291164220114, 'mae': 4.450327430394522, 'rmse': np.float64(6.001431716020497)}, 'best_params': {'model__max_depth': 8, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 2, 'model__min_samples_split': 2, 'model__n_estimators': 200}})


-----
#### 재배치 기준

In [224]:
# 04:00 / 16:00 기준으로 2번 묶기 + 날씨(Open-Meteo 일별 평균)
import pandas as pd
import requests

def fetch_daily_weather_open_meteo(
    start_date,
    end_date,
    lat,
    lon,
    timezone="Asia/Seoul",
    cache=None,
):
    if cache is None:
        cache = {}

    key = (start_date, end_date, lat, lon, timezone)
    if key in cache:
        return cache[key].copy()

    url = (
        "https://archive-api.open-meteo.com/v1/archive"
        "?latitude={lat}&longitude={lon}"
        "&start_date={start}&end_date={end}"
        "&hourly=temperature_2m,precipitation,wind_speed_10m,snowfall"
        "&timezone={tz}"
    ).format(lat=lat, lon=lon, start=start_date, end=end_date, tz=timezone)

    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    hourly = data.get("hourly", {})
    df = pd.DataFrame({
        "time": hourly.get("time", []),
        "temperature_2m": hourly.get("temperature_2m", []),
        "precipitation": hourly.get("precipitation", []),
        "wind_speed_10m": hourly.get("wind_speed_10m", []),
        "snowfall": hourly.get("snowfall", []),
    })

    if df.empty:
        daily = pd.DataFrame(columns=["date", "기온", "강수량", "풍속", "적설량"])
    else:
        dt = pd.to_datetime(df["time"])
        df["date"] = dt.dt.date
        daily = (
            df.groupby("date", as_index=False)
              .agg({
                  "temperature_2m": "mean",
                  "precipitation": "mean",
                  "wind_speed_10m": "mean",
                  "snowfall": "mean",
              })
        )
        daily = daily.rename(columns={
            "temperature_2m": "기온",
            "precipitation": "강수량",
            "wind_speed_10m": "풍속",
            "snowfall": "적설량",
        })

    cache[key] = daily
    return daily.copy()

def aggregate_by_shift_4am_4pm(
    df,
    time_col="날씨시각",
    sum_cols=("상대대수",),
    keep_cols=("주소1", "요일", "주말", "출퇴근시간", "계절", "시각"),
    lat=37.60279,
    lon=126.92918,
    timezone="Asia/Seoul",
    weather_cache=None,
):
    out = df.copy()
    dt = pd.to_datetime(out[time_col])
    hour = dt.dt.hour

    # 버킷 라벨: 04:00, 16:00 (16:00~03:59는 다음날 04:00으로)
    base_date = dt.dt.normalize()
    bucket_time = pd.Series(index=out.index, dtype="datetime64[ns]")
    # 00:00~03:59 -> 당일 04:00
    mask_early = hour < 4
    bucket_time.loc[mask_early] = base_date[mask_early] + pd.Timedelta(hours=4)
    # 04:00~15:59 -> 당일 16:00
    mask_day = (hour >= 4) & (hour < 16)
    bucket_time.loc[mask_day] = base_date[mask_day] + pd.Timedelta(hours=16)
    # 16:00~23:59 -> 다음날 04:00
    mask_evening = hour >= 16
    bucket_time.loc[mask_evening] = base_date[mask_evening] + pd.Timedelta(days=1, hours=4)
    out[time_col] = bucket_time

    agg_map = {}
    for c in sum_cols:
        if c in out.columns:
            agg_map[c] = "sum"
    for c in keep_cols:
        if c in out.columns:
            agg_map[c] = "first"

    grouped = (
        out.groupby(time_col, as_index=False)
           .agg(agg_map)
           .sort_values(time_col)
           .reset_index(drop=True)
    )

    # Open-Meteo 일별 평균 날씨 붙이기
    grouped["date"] = pd.to_datetime(grouped[time_col]).dt.date
    start_date = str(grouped["date"].min())
    end_date = str(grouped["date"].max())
    daily_weather = fetch_daily_weather_open_meteo(
        start_date,
        end_date,
        lat=lat,
        lon=lon,
        timezone=timezone,
        cache=weather_cache,
    )
    grouped = grouped.merge(daily_weather, how="left", on="date")
    if "date" in grouped.columns:
        grouped = grouped.drop(columns=["date"])

    return grouped

weather_cache = {}
all_2247_shift = aggregate_by_shift_4am_4pm(all_2247, weather_cache=weather_cache)
all_2252_shift = aggregate_by_shift_4am_4pm(all_2252, weather_cache=weather_cache)
all_2425_shift = aggregate_by_shift_4am_4pm(all_2425, weather_cache=weather_cache)


In [225]:
all_2247_shift.to_csv("../Data/ST-2247_current2.csv", index=False)
all_2252_shift.to_csv("../Data/ST-2252_current2.csv", index=False)
all_2425_shift.to_csv("../Data/ST-2425_current2.csv", index=False)

In [226]:
all_2247_shift

,날씨시각,상대대수,기온,강수량,적설량,풍속,주소1,요일,주말,계절
0,2024-01-01 04:00:00,-2.0,-1.900000,0.000000,0.0,6.300000,서울특별시 은평구 신사동 178-9,월,1,4
1,2024-01-01 16:00:00,0.0,1.800000,0.000000,0.0,8.500000,서울특별시 은평구 신사동 178-9,월,1,4
2,2024-01-02 04:00:00,8.0,1.400000,0.016667,0.0,7.350000,서울특별시 은평구 연서로 124,월,1,4
3,2024-01-02 16:00:00,-12.0,1.437500,0.000000,0.0,5.662500,서울특별시 은평구 신사동 178-9,화,0,4
4,2024-01-03 04:00:00,6.0,0.237500,0.000000,0.0,6.487500,서울특별시 은평구 증산동 199-8,화,0,4
...,...,...,...,...,...,...,...,...,...,...
713,2024-12-30 04:00:00,-2.0,0.600000,0.000000,0.0,4.200000,서울특별시 은평구 역촌동 45-34,일,1,4
714,2024-12-30 16:00:00,-6.0,4.333333,0.016667,0.0,10.066667,서울특별시 은평구 신사동 178-9,월,0,4
715,2024-12-31 04:00:00,0.0,2.000000,0.016667,0.0,10.816667,서울특별시 은평구 증산동 199-8,월,0,4
716,2024-12-31 16:00:00,0.0,-0.142857,0.000000,0.0,12.142857,서울특별시 은평구 신사동 352-2,화,0,4


In [227]:
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def train_rf_with_search_time_split_2(
    df,
    time_col="날씨시각",
    feature_cols=("주말", "계절", "기온", "강수량", "적설량", "풍속"),
    target_col="상대대수",
    test_size=0.2,
    val_size=0.2,
    random_state=42,
):
    # 1) 시간순 정렬
    df_sorted = df.sort_values(time_col).reset_index(drop=True)

    X = df_sorted[list(feature_cols)]
    y = df_sorted[target_col]

    n = len(df_sorted)
    n_test = int(n * test_size)
    n_val = int(n * val_size)

    n_train = n - n_val - n_test
    if n_train <= 0:
        raise ValueError("train/val/test 비율이 너무 큽니다.")

    # 2) 시간순 분할
    X_train, y_train = X.iloc[:n_train], y.iloc[:n_train]
    X_val, y_val = X.iloc[n_train:n_train+n_val], y.iloc[n_train:n_train+n_val]
    X_test, y_test = X.iloc[n_train+n_val:], y.iloc[n_train+n_val:]

    # 3) 파이프라인 (정규화 + 랜덤포레스트)
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", RandomForestRegressor(random_state=random_state, n_jobs=-1))
    ])

    # 4) 하이퍼파라미터 탐색
    param_grid = {
        "model__n_estimators": [200, 400],
        "model__max_depth": [None, 8, 12],
        "model__min_samples_split": [2, 5],
        "model__min_samples_leaf": [1, 2],
        "model__max_features": ["sqrt", "log2"]
    }

    search = GridSearchCV(
        pipe,
        param_grid=param_grid,
        cv=3,
        scoring="r2",
        n_jobs=-1
    )
    search.fit(X_train, y_train)

    best_model = search.best_estimator_

    # 5) 점수 출력
    def _eval(name, X_split, y_split):
        pred = best_model.predict(X_split)
        return {
            "split": name,
            "r2": r2_score(y_split, pred),
            "mae": mean_absolute_error(y_split, pred),
            "rmse": np.sqrt(mean_squared_error(y_split, pred))
        }

    scores = {
        "train": _eval("train", X_train, y_train),
        "val": _eval("val", X_val, y_val),
        "test": _eval("test", X_test, y_test),
        "best_params": search.best_params_
    }

    return best_model, scores


In [228]:
print(train_rf_with_search_time_split_2(all_2247_shift))

(Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 RandomForestRegressor(max_depth=8, max_features='log2',
                                       min_samples_leaf=2, min_samples_split=5,
                                       n_estimators=200, n_jobs=-1,
                                       random_state=42))]), {'train': {'split': 'train', 'r2': 0.43086072186244306, 'mae': 4.71540465793994, 'rmse': np.float64(6.4173945499823075)}, 'val': {'split': 'val', 'r2': -0.03472636425293296, 'mae': 7.927385007047941, 'rmse': np.float64(10.496676501199993)}, 'test': {'split': 'test', 'r2': -0.08769048305089933, 'mae': 5.7002847498580715, 'rmse': np.float64(7.539675573505791)}, 'best_params': {'model__max_depth': 8, 'model__max_features': 'log2', 'model__min_samples_leaf': 2, 'model__min_samples_split': 5, 'model__n_estimators': 200}})
